# IDRiD Disease Grading — EDA & Ben Graham Preprocessing

**Purpose (Phase C):** Preprocess the IDRiD Disease Grading fundus images into the same 512×512 PNG format
expected by the APTOS pipeline, so they can supplement Grade 3+4 training data.

**Pipeline:**
1. EDA — Understand the raw dataset: labels, images, distribution
2. Domain Shift Analysis — Compare IDRiD vs APTOS after preprocessing
3. Preprocessing — Apply Ben Graham local-contrast normalization (same `src/dataset.py` function)
4. Output — Save PNGs + write `data/idrid_labels.csv` in `id_code,diagnosis` format

---
**Note:** All actual disk writes happen in Section 4 (`PREPROCESSING RUN`).  
Sections 1–3 are read-only EDA and preview.

## 0 — Setup & Paths

In [ ]:
from __future__ import annotations

import csv
import multiprocessing
import os
import sys
import warnings
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ── project root on path so src.* imports work ──────────────────────────────
ROOT_DIR = Path(".").resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.dataset import ben_graham_preprocess

# ── IDRiD raw inputs ─────────────────────────────────────────────────────────
IDRID_DIR         = ROOT_DIR / "B_Disease_Grading"
TRAIN_IMG_DIR     = IDRID_DIR / "1. Original Images" / "a. Training Set"
TEST_IMG_DIR      = IDRID_DIR / "1. Original Images" / "b. Testing Set"
TRAIN_CSV         = IDRID_DIR / "2. Groundtruths" / "a. IDRiD_Disease Grading_Training Labels.csv"
TEST_CSV          = IDRID_DIR / "2. Groundtruths" / "b. IDRiD_Disease Grading_Testing Labels.csv"
IMG_EXT           = ".jpg"

# ── Output ──────────────────────────────────────────────────────────────────
OUTPUT_IMG_DIR    = ROOT_DIR / "data" / "idrid_processed"
OUTPUT_CSV        = ROOT_DIR / "data" / "idrid_labels.csv"
FAILED_LOG        = ROOT_DIR / "data" / "idrid_failed.txt"
IMAGE_SIZE        = 512
NUM_WORKERS       = max(1, multiprocessing.cpu_count() - 1)

# ── APTOS reference for comparison ──────────────────────────────────────────
APTOS_TRAIN_CSV   = ROOT_DIR / "data" / "data_split" / "train_label.csv"
APTOS_IMG_DIR     = ROOT_DIR / "data" / "data_split" / "train_split"

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
CLASS_COLORS = ["#4CAF50", "#8BC34A", "#FFC107", "#FF5722", "#F44336"]

# ── Grades to include in C1 supplement ──────────────────────────────────────
SELECTED_GRADES = {3, 4}

print(f"ROOT_DIR     : {ROOT_DIR}")
print(f"TRAIN_IMG_DIR: {TRAIN_IMG_DIR}")
print(f"TEST_IMG_DIR : {TEST_IMG_DIR}")
print(f"NUM_WORKERS  : {NUM_WORKERS}")
print(f"cv2 version  : {cv2.__version__}")

## 1 — EDA: Labels

### 1.1 — Load and inspect the CSVs

In [ ]:
# Read IDRiD CSVs — note: columns have inconsistent spacing
df_train_raw = pd.read_csv(TRAIN_CSV)
df_test_raw  = pd.read_csv(TEST_CSV)

print(f"Train CSV shape : {df_train_raw.shape}")
print(f"Train columns   : {df_train_raw.columns.tolist()}")
print(f"Test CSV shape  : {df_test_raw.shape}")
print(f"Test columns    : {df_test_raw.columns.tolist()}")

df_train_raw.head(5)

In [ ]:
# Standardize column names
def clean_idrid_df(df, partition, img_dir):
    df = df.copy()
    # IDRiD CSVs have columns like 'Image name', 'Retinopathy grade', 'Risk of macular edema '
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={
        'Image name': 'id_code_orig',
        'Retinopathy grade': 'diagnosis',
        'Risk of macular edema': 'dme_risk',
    })
    df['diagnosis'] = df['diagnosis'].astype(int)
    if 'dme_risk' in df.columns:
        df['dme_risk'] = df['dme_risk'].astype(int)
    df['partition'] = partition
    df['id_code'] = df['id_code_orig'].apply(lambda x: f"idrid_{partition}_{x.strip()}")
    df['img_path'] = df['id_code_orig'].apply(lambda x: img_dir / f"{x.strip()}{IMG_EXT}")
    return df

df_train = clean_idrid_df(df_train_raw, 'train', TRAIN_IMG_DIR)
df_test  = clean_idrid_df(df_test_raw,  'test',  TEST_IMG_DIR)

df = pd.concat([df_train, df_test], ignore_index=True)

print(f"Total records         : {len(df):,}")
print(f"Unique image IDs      : {df['id_code'].nunique():,}")
print(f"Duplicate entries     : {df.duplicated('id_code').sum()}")
print(f"Null values           : {df[['id_code', 'diagnosis']].isnull().sum().to_dict()}")
print()
print("First 5 rows:")
df[['id_code', 'diagnosis', 'partition']].head()

### 1.2 — Class distribution

In [ ]:
counts = df["diagnosis"].value_counts().sort_index()
print("IDRiD class distribution (all partitions):")
for cls, name in enumerate(CLASS_NAMES):
    n = counts.get(cls, 0)
    pct = 100 * n / len(df)
    bar = "█" * int(pct / 2)
    print(f"  [{cls}] {name:<18} {n:>6,}  ({pct:5.1f}%)  {bar}")
print(f"  {'TOTAL':<22} {len(df):>6,}")

In [ ]:
# ── Side-by-side: IDRiD vs APTOS class distributions ─────────────────────
df_aptos = pd.read_csv(APTOS_TRAIN_CSV)
aptos_counts = df_aptos["diagnosis"].value_counts().sort_index()

x = np.arange(5)
w = 0.38

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Absolute counts
ax = axes[0]
bars1 = ax.bar(x - w/2, counts.values, w, label="IDRiD", color=CLASS_COLORS, alpha=0.85, edgecolor="black", linewidth=0.6)
bars2 = ax.bar(x + w/2, aptos_counts.values, w, label="APTOS 2019", color=CLASS_COLORS, alpha=0.45, edgecolor="black", linewidth=0.6, hatch="//")
ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n{c}" for n, c in enumerate(CLASS_NAMES)], fontsize=9)
ax.set_ylabel("Image count")
ax.set_title("Absolute class counts — IDRiD vs APTOS")
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=7)

# Proportions
ax2 = axes[1]
idrid_pct = counts.values / counts.values.sum() * 100
aptos_pct = aptos_counts.values / aptos_counts.values.sum() * 100
ax2.bar(x - w/2, idrid_pct, w, label="IDRiD", color=CLASS_COLORS, alpha=0.85, edgecolor="black", linewidth=0.6)
ax2.bar(x + w/2, aptos_pct, w, label="APTOS 2019", color=CLASS_COLORS, alpha=0.45, edgecolor="black", linewidth=0.6, hatch="//")
ax2.set_xticks(x)
ax2.set_xticklabels([f"{n}\n{c}" for n, c in enumerate(CLASS_NAMES)], fontsize=9)
ax2.set_ylabel("% of dataset")
ax2.set_title("Class proportions — IDRiD vs APTOS")
ax2.legend()

plt.tight_layout()
plt.savefig(ROOT_DIR / "results" / "idrid_class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: results/idrid_class_distribution.png")

### 1.3 — Macular edema grade analysis

In [ ]:
if 'dme_risk' in df.columns:
    ct = pd.crosstab(df['diagnosis'], df['dme_risk'],
                     rownames=['DR Grade'], colnames=['DME Risk'])
    print("Cross-tabulation: DR Grade × DME Risk")
    print(ct)

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(ct.values, cmap="YlOrRd")
    ax.set_xticks(range(ct.shape[1]))
    ax.set_yticks(range(ct.shape[0]))
    ax.set_xticklabels([f"DME {c}" for c in ct.columns], fontsize=8)
    ax.set_yticklabels([f"{i} {CLASS_NAMES[i]}" for i in ct.index], fontsize=8)
    ax.set_title("DR Grade vs DME Risk (IDRiD)")
    plt.colorbar(im, ax=ax)
    for i in range(ct.shape[0]):
        for j in range(ct.shape[1]):
            val = ct.values[i, j]
            ax.text(j, i, str(val), ha="center", va="center", fontsize=8,
                    color="white" if val > ct.values.max() * 0.6 else "black")
    plt.tight_layout()
    plt.savefig(ROOT_DIR / "results" / "idrid_dr_dme_crosstab.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("DME risk column not found")

## 2 — EDA: Image Files

### 2.1 — File existence audit

In [ ]:
df["img_exists"] = df["img_path"].apply(lambda p: Path(p).exists())

n_found   = df["img_exists"].sum()
n_missing = (~df["img_exists"]).sum()
print(f"CSV rows        : {len(df):,}")
print(f"Images found    : {n_found:,}")
print(f"Images MISSING  : {n_missing:,}")

if n_missing > 0:
    print("\nFirst 10 missing IDs:")
    print(df[~df["img_exists"]]["id_code"].head(10).tolist())

df_valid = df[df["img_exists"]].reset_index(drop=True)
print(f"\nUsable rows for preprocessing: {len(df_valid):,}")

### 2.2 — Raw image dimensions and file sizes

In [ ]:
# Sample images to check dimensions and file sizes
sample_n = min(100, len(df_valid))
sample_df = df_valid.sample(sample_n, random_state=42).reset_index(drop=True)

heights, widths, filesizes_kb = [], [], []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Sampling images"):
    p = Path(row["img_path"])
    filesizes_kb.append(p.stat().st_size / 1024)
    img = cv2.imread(str(p))
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h)
        widths.append(w)

print(f"\nHeight  — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"Width   — min: {min(widths)},  max: {max(widths)},  mean: {np.mean(widths):.0f}")
print(f"File KB — min: {min(filesizes_kb):.1f}, max: {max(filesizes_kb):.1f}, mean: {np.mean(filesizes_kb):.1f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].hist(heights, bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Image heights (px)")
axes[1].hist(widths, bins=20, color="coral", edgecolor="white")
axes[1].set_title("Image widths (px)")
axes[2].hist(filesizes_kb, bins=20, color="mediumpurple", edgecolor="white")
axes[2].set_title("File size (KB)")
plt.tight_layout()
plt.savefig(ROOT_DIR / "results" / "idrid_raw_dimensions.png", dpi=120, bbox_inches="tight")
plt.show()

### 2.3 — Raw sample images by grade

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(12, 20))

for cls in range(5):
    grade_df = df_valid[df_valid["diagnosis"] == cls]
    samples = grade_df.sample(min(3, len(grade_df)), random_state=42)
    for j, (_, row) in enumerate(samples.iterrows()):
        img = cv2.imread(str(row["img_path"]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[cls][j].imshow(img)
        axes[cls][j].set_title(f"Grade {cls} — {CLASS_NAMES[cls]}\n{row['id_code_orig']}", fontsize=8)
        axes[cls][j].axis("off")

plt.suptitle("IDRiD: Raw sample images by DR grade", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(ROOT_DIR / "results" / "idrid_raw_samples.png", dpi=120, bbox_inches="tight")
plt.show()

## 3 — Domain Shift Analysis (APTOS vs IDRiD)

### 3.1 — Pixel intensity distribution comparison (after preprocessing)

In [ ]:
# Compare RGB histograms of preprocessed APTOS vs IDRiD images
def get_rgb_histogram(img_path, size=IMAGE_SIZE):
    """Read an image, apply Ben Graham preprocessing, return RGB histograms."""
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = ben_graham_preprocess(img, size)
    hists = []
    for c in range(3):
        hist, _ = np.histogram(img[:, :, c], bins=256, range=(0, 256))
        hists.append(hist)
    return hists

# Sample from IDRiD (Grade 3+4)
idrid_grade34 = df_valid[df_valid["diagnosis"].isin(SELECTED_GRADES)]
idrid_sample = idrid_grade34.sample(min(30, len(idrid_grade34)), random_state=42)

# Sample from APTOS (Grade 3+4)
df_aptos_full = pd.read_csv(APTOS_TRAIN_CSV)
aptos_grade34 = df_aptos_full[df_aptos_full["diagnosis"].isin(SELECTED_GRADES)]
aptos_sample = aptos_grade34.sample(min(30, len(aptos_grade34)), random_state=42)

# Collect histograms
idrid_hists_r, idrid_hists_g, idrid_hists_b = [], [], []
for _, row in tqdm(idrid_sample.iterrows(), total=len(idrid_sample), desc="IDRiD histograms"):
    h = get_rgb_histogram(row["img_path"])
    if h:
        idrid_hists_r.append(h[0])
        idrid_hists_g.append(h[1])
        idrid_hists_b.append(h[2])

aptos_hists_r, aptos_hists_g, aptos_hists_b = [], [], []
for _, row in tqdm(aptos_sample.iterrows(), total=len(aptos_sample), desc="APTOS histograms"):
    img_path = APTOS_IMG_DIR / f"{row['id_code']}.png"
    h = get_rgb_histogram(img_path)
    if h:
        aptos_hists_r.append(h[0])
        aptos_hists_g.append(h[1])
        aptos_hists_b.append(h[2])

# Plot mean histograms
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channel_names = ["Red", "Green", "Blue"]
channel_colors = ["red", "green", "blue"]
for i, (idrid_h, aptos_h) in enumerate([
    (idrid_hists_r, aptos_hists_r),
    (idrid_hists_g, aptos_hists_g),
    (idrid_hists_b, aptos_hists_b),
]):
    ax = axes[i]
    mean_idrid = np.mean(idrid_h, axis=0)
    mean_aptos = np.mean(aptos_h, axis=0)
    ax.plot(mean_idrid, color=channel_colors[i], alpha=0.9, label="IDRiD (Grade 3+4)", linewidth=1.5)
    ax.plot(mean_aptos, color=channel_colors[i], alpha=0.4, linestyle="--", label="APTOS (Grade 3+4)", linewidth=1.5)
    ax.set_title(f"{channel_names[i]} channel")
    ax.set_xlabel("Pixel value")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=7)

plt.suptitle("Preprocessed pixel intensity: IDRiD vs APTOS (Grade 3+4)", fontsize=12)
plt.tight_layout()
plt.savefig(ROOT_DIR / "results" / "idrid_vs_aptos_pixel_histograms.png", dpi=120, bbox_inches="tight")
plt.show()

### 3.2 — Preprocessed sample grid comparison

In [ ]:
# Show Ben Graham preprocessed Grade 3+4 samples side by side
fig, axes = plt.subplots(2, 6, figsize=(18, 6))

# IDRiD Grade 3+4 (top row)
idrid_samples = idrid_grade34.sample(min(6, len(idrid_grade34)), random_state=123)
for j, (_, row) in enumerate(idrid_samples.iterrows()):
    if j >= 6:
        break
    img = cv2.imread(str(row["img_path"]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = ben_graham_preprocess(img, IMAGE_SIZE)
    axes[0][j].imshow(img)
    axes[0][j].set_title(f"IDRiD G{row['diagnosis']}\n{row['id_code_orig']}", fontsize=7)
    axes[0][j].axis("off")

# APTOS Grade 3+4 (bottom row)
aptos_samples = aptos_grade34.sample(min(6, len(aptos_grade34)), random_state=123)
for j, (_, row) in enumerate(aptos_samples.iterrows()):
    if j >= 6:
        break
    img_path = APTOS_IMG_DIR / f"{row['id_code']}.png"
    img = cv2.imread(str(img_path))
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = ben_graham_preprocess(img, IMAGE_SIZE)
        axes[1][j].imshow(img)
        axes[1][j].set_title(f"APTOS G{row['diagnosis']}\n{row['id_code']}", fontsize=7)
    axes[1][j].axis("off")

axes[0][0].set_ylabel("IDRiD", fontsize=12, fontweight="bold")
axes[1][0].set_ylabel("APTOS", fontsize=12, fontweight="bold")
plt.suptitle("Ben Graham preprocessed — Grade 3+4 comparison", fontsize=13)
plt.tight_layout()
plt.savefig(ROOT_DIR / "results" / "idrid_vs_aptos_preprocessed_samples.png", dpi=120, bbox_inches="tight")
plt.show()

### 3.3 — Feature-space domain shift (t-SNE)

In [ ]:
import torch
import torchvision.models as models
from sklearn.manifold import TSNE

# Load a pretrained ResNet-50 and use it as a feature extractor
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = torch.nn.Identity()  # remove classification head → 2048-d features
model = model.to(device)
model.eval()

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def extract_features(img_path, model, device, size=IMAGE_SIZE):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = ben_graham_preprocess(img, size)
    t = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
    for c in range(3):
        t[c] = (t[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
    t = t.unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model(t)
    return feat.cpu().numpy().flatten()

# Extract features from IDRiD Grade 3+4
features, labels_list, sources = [], [], []
for _, row in tqdm(idrid_grade34.iterrows(), total=len(idrid_grade34), desc="IDRiD features"):
    f = extract_features(row["img_path"], model, device)
    if f is not None:
        features.append(f)
        labels_list.append(row["diagnosis"])
        sources.append("IDRiD")

# Extract features from APTOS Grade 3+4
for _, row in tqdm(aptos_grade34.iterrows(), total=len(aptos_grade34), desc="APTOS features"):
    img_path = APTOS_IMG_DIR / f"{row['id_code']}.png"
    f = extract_features(img_path, model, device)
    if f is not None:
        features.append(f)
        labels_list.append(row["diagnosis"])
        sources.append("APTOS")

features = np.array(features)
print(f"Total features extracted: {len(features)} ({sum(1 for s in sources if s=='IDRiD')} IDRiD, {sum(1 for s in sources if s=='APTOS')} APTOS)")

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embeddings = tsne.fit_transform(features)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Colored by dataset
ax = axes[0]
for src, color, marker in [("IDRiD", "tab:blue", "o"), ("APTOS", "tab:orange", "s")]:
    mask = [s == src for s in sources]
    ax.scatter(embeddings[mask, 0], embeddings[mask, 1], c=color, marker=marker,
              label=src, alpha=0.6, s=30, edgecolors="white", linewidth=0.3)
ax.set_title("t-SNE: Colored by Dataset")
ax.legend()
ax.set_xticks([])
ax.set_yticks([])

# Colored by grade
ax2 = axes[1]
for grade, color in [(3, "#FF5722"), (4, "#F44336")]:
    mask = [l == grade for l in labels_list]
    ax2.scatter(embeddings[mask, 0], embeddings[mask, 1], c=color,
               label=f"Grade {grade} ({CLASS_NAMES[grade]})",
               alpha=0.6, s=30, edgecolors="white", linewidth=0.3)
ax2.set_title("t-SNE: Colored by DR Grade")
ax2.legend()
ax2.set_xticks([])
ax2.set_yticks([])

plt.suptitle("Feature-space domain shift: IDRiD vs APTOS (Grade 3+4)", fontsize=13)
plt.tight_layout()
plt.savefig(ROOT_DIR / "results" / "idrid_vs_aptos_tsne.png", dpi=120, bbox_inches="tight")
plt.show()
print("If IDRiD and APTOS points overlap → low domain shift → safe to supplement.")

## 4 — Preprocessing Run

### 4.1 — Grade filter and expected output

In [ ]:
df_selected = df_valid[df_valid["diagnosis"].isin(SELECTED_GRADES)].reset_index(drop=True)

print(f"Selected grades: {sorted(SELECTED_GRADES)}")
print(f"Total images to preprocess: {len(df_selected)}")
print()
for g in sorted(SELECTED_GRADES):
    n = (df_selected["diagnosis"] == g).sum()
    print(f"  Grade {g} ({CLASS_NAMES[g]}): {n} images")

### 4.2 — Batch preprocessing

In [ ]:
OUTPUT_IMG_DIR.mkdir(parents=True, exist_ok=True)

processed = []
failed = []

for _, row in tqdm(df_selected.iterrows(), total=len(df_selected), desc="Preprocessing"):
    try:
        img = cv2.imread(str(row["img_path"]))
        if img is None:
            failed.append(row["id_code"])
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = ben_graham_preprocess(img, IMAGE_SIZE)

        out_path = OUTPUT_IMG_DIR / f"{row['id_code']}.png"
        cv2.imwrite(str(out_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

        processed.append({
            "id_code": row["id_code"],
            "diagnosis": row["diagnosis"],
        })
    except Exception as e:
        print(f"  ERROR processing {row['id_code']}: {e}")
        failed.append(row["id_code"])

print(f"\n✓ Preprocessed: {len(processed)} images")
print(f"✗ Failed: {len(failed)} images")

if failed:
    with open(FAILED_LOG, "w") as f:
        f.write("\n".join(failed))
    print(f"  Failed IDs written to {FAILED_LOG}")

### 4.3 — Write output CSV

In [ ]:
df_out = pd.DataFrame(processed)
df_out = df_out.sort_values("id_code").reset_index(drop=True)
df_out.to_csv(OUTPUT_CSV, index=False)

print(f"✓ Written {len(df_out)} labels to {OUTPUT_CSV}")
print(f"  Columns: {df_out.columns.tolist()}")
print()
print("Grade distribution in output CSV:")
for g in sorted(SELECTED_GRADES):
    n = (df_out["diagnosis"] == g).sum()
    print(f"  Grade {g}: {n} images")
print()
df_out.head()

### 4.4 — Verification

In [ ]:
# Verify processed images
output_files = list(OUTPUT_IMG_DIR.glob("*.png"))
print(f"PNG files in output directory: {len(output_files)}")
print(f"Labels in CSV: {len(df_out)}")
assert len(output_files) == len(df_out), "Mismatch between images and labels!"

# Show a few processed images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i >= len(df_out):
        break
    row = df_out.iloc[i]
    img = cv2.imread(str(OUTPUT_IMG_DIR / f"{row['id_code']}.png"))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"Grade {row['diagnosis']}\n{row['id_code']}", fontsize=8)
    ax.axis("off")

plt.suptitle("Verification: Preprocessed IDRiD images", fontsize=13)
plt.tight_layout()
plt.savefig(ROOT_DIR / "results" / "idrid_preprocessed_verification.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n✓ Preprocessing complete. Ready for experiment C1 (exp_id=301).")